# Credit Card Fraud Detection

This is the second of two capstones I built after finishing Andrew Ng's Machine Learning Specialization. The first one (forest cover type) was a roughly balanced multiclass problem and let me practise everything from logistic regression up to gradient-boosted trees. The point of this one is the opposite: a binary problem with an extreme class imbalance, where most of the work is figuring out the right metrics and the right threshold instead of the right model.

Dataset: the well-known ULB credit-card fraud set — 284,807 transactions over two days in September 2013, of which 492 (~0.17%) are fraudulent. Most features (V1..V28) are anonymised PCA components produced by the original authors; only `Time` and `Amount` are raw. I pull it from OpenML so the download doesn't need a Kaggle account.

## 1. Load and split

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data
import models
import plots

X_train, y_train, X_val, y_val, X_test, y_test = data.load(seed=0)
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape}")
print(f"positives — train {y_train.sum()}, val {y_val.sum()}, test {y_test.sum()}")

## 2. A look at the data

In [ ]:
plots.class_balance(y_train)
plt.title("class balance, training set")
plt.show()

About 1 fraud per 580 legitimate transactions. Accuracy is useless here — predicting *legit* every time would already score 99.83%. I'll lean on precision, recall, F1, and the precision-recall curve instead.

In [ ]:
# Amount is the only non-anonymised feature in this mirror; pick a couple
# of V-columns that look most informative on the standard correlations
df = pd.DataFrame(np.c_[X_train, y_train], columns=data.FEATURE_NAMES + ["y"])
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, col in zip(axes, ["Amount", "V14", "V17"]):
    for c, name, color in [(0, "legit", "#4c78a8"), (1, "fraud", "#e45756")]:
        ax.hist(df.loc[df.y == c, col], bins=60, alpha=0.5, label=name,
                color=color, density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

Fraud transactions sit in noticeably different regions of V14 and V17 — those two are the strongest single-feature separators in this set. `Amount` is scaled in `data.load` so the from-scratch logistic regression doesn't blow up on the raw values.

## 3. Logistic regression from scratch (with class weights)

Plain mini-batch SGD, L2 regularisation, and a per-sample weight that's inversely proportional to class frequency — without it the gradient is dominated by the legitimate transactions and the model basically learns to always predict 0.

In [ ]:
clf_scratch = models.LogisticRegressionWeighted(
    lr=0.05, l2=1e-4, epochs=15, batch=512, seed=0,
).fit(X_train, y_train, X_val, y_val)

scores_scratch = clf_scratch.predict_proba(X_val)
print(f"AP on val: {__import__('sklearn.metrics', fromlist=['average_precision_score']).average_precision_score(y_val, scores_scratch):.3f}")

## 4. Logistic regression with sklearn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

clf_sk = LogisticRegression(
    class_weight="balanced", max_iter=1000, solver="lbfgs", n_jobs=-1,
).fit(X_train, y_train)

scores_sk = clf_sk.predict_proba(X_val)[:, 1]
print(f"AP on val: {average_precision_score(y_val, scores_sk):.3f}")

## 5. Small NN in Keras

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
tf.random.set_seed(0)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
class_weight = {0: len(y_train) / (2 * n_neg), 1: len(y_train) / (2 * n_pos)}

nn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu", kernel_initializer="he_normal"),
    tf.keras.layers.Dense(16, activation="relu", kernel_initializer="he_normal"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
nn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
nn.fit(X_train, y_train, validation_data=(X_val, y_val),
       epochs=10, batch_size=512, class_weight=class_weight, verbose=0)

scores_nn = nn.predict(X_val, verbose=0).ravel()
print(f"AP on val: {average_precision_score(y_val, scores_nn):.3f}")

## 6. Histogram gradient boosting

sklearn's `HistGradientBoostingClassifier` handles the imbalance via `class_weight='balanced'`. Fast, and a strong baseline on tabular data.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    max_iter=200, class_weight="balanced", random_state=0,
).fit(X_train, y_train)

scores_hgb = hgb.predict_proba(X_val)[:, 1]
print(f"AP on val: {average_precision_score(y_val, scores_hgb):.3f}")

## 7. Threshold tuning

All four models above output scores; turning those into 0/1 decisions needs a threshold. The "right" threshold depends on the cost of a missed fraud vs a false alarm. Here I'll just maximise F1 on the validation set and take that as a sensible operating point.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (name, s) in zip(axes.flat, [
    ("logistic (scratch)", scores_scratch),
    ("logistic (sklearn)", scores_sk),
    ("Keras NN", scores_nn),
    ("hist gradient boosting", scores_hgb),
]):
    plots.pr_curve(y_val, s, name, ax=ax)
    ax.set_title(f"{name}, AP {average_precision_score(y_val, s):.3f}")
plt.tight_layout()
plt.show()

In [ ]:
thresholds = {}
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (name, s) in zip(axes.flat, [
    ("logistic (scratch)", scores_scratch),
    ("logistic (sklearn)", scores_sk),
    ("Keras NN", scores_nn),
    ("hist gradient boosting", scores_hgb),
]):
    _, t = plots.f1_vs_threshold(y_val, s, ax=ax)
    ax.set_title(f"F1 vs threshold — {name}")
    thresholds[name] = float(t)
plt.tight_layout()
plt.show()
thresholds

## 8. Gaussian anomaly detection from scratch

The unsupervised baseline from Part III: fit a Gaussian per feature on the *legitimate* transactions only, score every transaction by its log-density, and flag the lowest-density ones. Threshold (epsilon) is picked on the validation set by maximising F1, exactly the procedure in the lectures.

In [ ]:
anom = models.GaussianAnomalyDetector().fit(X_train[y_train == 0])
val_scores = anom.score(X_val)
eps, val_f1 = anom.select_epsilon(val_scores, y_val)
print(f"chosen epsilon: {eps:.2f}, val F1: {val_f1:.3f}")

F1 here is much lower than the supervised models — expected, because anomaly detection sees no fraud labels at training time. It's still useful as a sanity check: anything the Gaussian model flags is genuinely far from the bulk of legitimate behaviour.

## 9. Final comparison on the test set

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
)

def metrics_at(name, scores, threshold, score_for_ap=None):
    if score_for_ap is None:
        score_for_ap = scores
    pred = (scores >= threshold).astype(int) if threshold is not None else scores
    return {
        "model": name,
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "PR-AUC": average_precision_score(y_test, score_for_ap),
    }

test_scores = {
    "logistic (scratch)": clf_scratch.predict_proba(X_test),
    "logistic (sklearn)": clf_sk.predict_proba(X_test)[:, 1],
    "Keras NN": nn.predict(X_test, verbose=0).ravel(),
    "hist gradient boosting": hgb.predict_proba(X_test)[:, 1],
}

results = [
    metrics_at(name, s, thresholds[name]) for name, s in test_scores.items()
]

# anomaly detector uses its own (non-comparable) scoring direction
anom_test = anom.score(X_test)
pred_anom = (anom_test < eps).astype(int)
results.append({
    "model": "Gaussian anomaly (unsupervised)",
    "precision": precision_score(y_test, pred_anom),
    "recall": recall_score(y_test, pred_anom),
    "F1": f1_score(y_test, pred_anom),
    "PR-AUC": average_precision_score(y_test, -anom_test),
})

pd.DataFrame(results).round(3).sort_values("F1", ascending=False).reset_index(drop=True)

In [ ]:
# pick the best model and look at its confusion matrix on the test set
best = max(test_scores.items(),
           key=lambda kv: f1_score(y_test, (kv[1] >= thresholds[kv[0]]).astype(int)))
best_name, best_scores = best
best_pred = (best_scores >= thresholds[best_name]).astype(int)
plots.confusion(y_test, best_pred, f"{best_name}, test")
plt.tight_layout()
plt.show()

The supervised models all reach F1 in roughly the same range; the anomaly detector lags behind, which it should given how much less it knows. The big practical lesson — which matches the course — is that on a problem this skewed, picking the threshold matters as much as picking the model.